<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/02_backtest_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Backtest

Notebook `01` produced a score. This notebook asks whether the score is *worth*
anything: do higher-scoring names earn higher forward returns? The method is an
**event study** — on each rebalance date, score the universe, sort names into
buckets by score, and measure their return over a fixed horizon. Aggregated across
many dates, the result shows whether congressional-buying conviction precedes price
moves, and by how much.

### What the backtest reports

- **bucket returns** — average forward return by score bucket. A signal with edge
  produces a rising staircase from the lowest bucket to the highest.
- **information coefficient (IC)** — the rank correlation between score and forward
  return, averaged across dates. A small positive IC that holds in most periods is
  the hallmark of a real, if modest, signal.
- **long-short spread** — the top bucket's return minus the bottom's, each period.
- **equity curve and risk metrics** — a simple strategy that holds the top bucket,
  with Sharpe, drawdown, and hit rate.
- **feature attribution** — forward returns split by committee alignment, which is
  how a feature earns or loses its weight.


## Setup

We pull the latest repo state, install dependencies including yfinance, and read the
Quiver token from Secrets. The backtest defaults to a long **mock history**; set
`USE_LIVE = True` for real Quiver data.

Committee-sector resolution uses a built-in static map, so the backtest makes no
per-ticker network calls and will not hit yfinance rate limits.


In [2]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests yfinance

import subprocess, os, sys, logging
from google.colab import userdata

GITHUB_USER = "balla-a12"
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
logging.getLogger("yfinance").setLevel(logging.CRITICAL)   # quiet delisted-ticker notices
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True to backtest on live Quiver data

Working in: /content/quant-equity-research


## Write the project modules

This writes the backtest layer and the supporting updates: a tolerant congress
normalizer that handles both the recent and bulk Quiver schemas, a price layer that
maps share-class tickers (BRK.B to BRK-B) and drops unfetchable names, an enrichment
layer with static sectors, and an event-study engine that degrades gracefully when
data is thin.


In [3]:
open("src/quant_research/ingestion/mock_data.py", "w").write('"""Synthetic data shaped like the Quiver Quantitative API responses.\n\nColumn names and value formats mirror the live `quiverquant` package so the same\nnormalization works on mock and real data. A few tickers carry heavy, purchase-\nskewed activity, and some buys are routed to representatives whose committee aligns\nwith the traded sector, so the signal\'s features have structure to detect. Member\nnames are real, so the live enrichment layer resolves them directly.\n"""\nimport numpy as np\nimport pandas as pd\nfrom datetime import date, timedelta\n\nfrom ..enrichment import mock as menr\n\n_UNIVERSE = ["PLTR", "LMT", "RTX", "AXON", "NOC", "CELH", "MELI", "CAVA",\n             "NVDA", "AAPL", "MSFT", "GE", "BA", "CAT", "JPM"]\n_HOT = ["PLTR", "LMT", "AXON"]\n_RANGES = ["$1,001 - $15,000", "$15,001 - $50,000", "$50,001 - $100,000",\n           "$100,001 - $250,000", "$250,001 - $500,000", "$500,001 - $1,000,000"]\n_REPS = list(menr.REP_COMMITTEE.keys())\n_TITLES = ["CEO", "CFO", "Director", "President", "EVP", "VP Finance", "COO"]\n_AGENCIES = ["Department of Defense", "Department of Energy", "NASA",\n             "Department of Homeland Security", "Department of Health"]\n_ISSUES = ["Defense", "Healthcare", "Taxation", "Energy", "Technology", "Trade"]\n\n\ndef _recent(rng, max_days_ago, min_days_ago=0):\n    return date.today() - timedelta(days=int(rng.integers(min_days_ago, max_days_ago)))\n\n\ndef mock_congress_trading(seed=42, n=180, history_days=40):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        if rng.random() < 0.45:\n            ticker = rng.choice(_HOT)\n            txn = rng.choice(["Purchase", "Sale"], p=[0.85, 0.15])\n        else:\n            ticker = rng.choice(_UNIVERSE)\n            txn = rng.choice(["Purchase", "Sale"], p=[0.55, 0.45])\n\n        sector = menr.TICKER_SECTOR.get(ticker)\n        aligned = menr.SECTOR_REPS.get(sector, [])\n        rep = rng.choice(aligned) if (aligned and rng.random() < 0.5) else rng.choice(_REPS)\n\n        report = _recent(rng, history_days)\n        transaction = report - timedelta(days=int(rng.integers(10, 45)))\n        rows.append({\n            "Representative": rep,\n            "Party": "D" if _REPS.index(rep) % 2 == 0 else "R",\n            "House": rng.choice(["Representatives", "Senate"], p=[0.75, 0.25]),\n            "Ticker": ticker, "Transaction": txn, "Range": rng.choice(_RANGES),\n            "TransactionDate": transaction, "ReportDate": report,\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_insiders(seed=43, n=120):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_HOT) if rng.random() < 0.4 else rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Name": f"Insider {rng.integers(1000, 9999)}",\n            "Title": rng.choice(_TITLES), "Date": _recent(rng, 90),\n            "TransactionCode": rng.choice(["P", "S"], p=[0.6, 0.4]),\n            "Shares": int(rng.integers(500, 50000)),\n            "PricePerShare": round(float(rng.uniform(20, 400)), 2),\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_gov_contracts(seed=44, n=80):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_HOT) if rng.random() < 0.5 else rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Date": _recent(rng, 90),\n            "Amount": int(rng.integers(50_000, 500_000_000)),\n            "Agency": rng.choice(_AGENCIES), "Description": "Procurement contract award",\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_lobbying(seed=45, n=140):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Date": _recent(rng, 360),\n            "Amount": int(rng.integers(10_000, 5_000_000)),\n            "Client": f"{ticker} Inc.", "Issue": rng.choice(_ISSUES),\n        })\n    return pd.DataFrame(rows)\n')
open("src/quant_research/ingestion/client.py", "w").write('"""A unified client for Quiver Quantitative data.\n\nThe QuiverClient exposes one method per dataset. Each method fetches raw data\n(from the live API when a token is supplied, otherwise from the mock generator)\nand returns it normalized into a consistent internal schema. The rest of the\nproject depends only on that internal schema, so nothing downstream needs to\nknow or care whether the data came from the live API or the mock source.\n"""\nimport re\nimport pandas as pd\n\nfrom . import mock_data\n\n\ndef _parse_range(text):\n    """Turn a Quiver amount range like \'$15,001 - $50,000\' into (low, high)."""\n    nums = re.findall(r"[\\d,]+", str(text))\n    vals = [int(n.replace(",", "")) for n in nums if n.replace(",", "").isdigit()]\n    if len(vals) >= 2:\n        return vals[0], vals[1]\n    if len(vals) == 1:\n        return vals[0], vals[0]\n    return 0, 0\n\n\nclass QuiverClient:\n    def __init__(self, token=None, mock=False, mock_history_days=40):\n        self.mock_history_days = mock_history_days\n        # No token means we cannot reach the live API, so we fall back to mock.\n        self.mock = mock or token is None\n        self._api = None\n        if not self.mock:\n            import quiverquant            # imported lazily; unused in mock mode\n            self._api = quiverquant.quiver(token)\n\n    # ---- Congressional trades -------------------------------------------\n    def congress_trades(self, historical=False):\n        if self.mock:\n            raw = mock_data.mock_congress_trading(\n                history_days=self.mock_history_days,\n                n=max(180, self.mock_history_days * 4))\n        else:\n            raw = self._api.congress_trading(recent=not historical)\n        return self._normalize_congress(raw)\n\n    @staticmethod\n    def _col(df, *names, default=""):\n        """First present column among `names`, else a default-filled Series.\n\n        The recent and bulk Quiver endpoints differ in their column names, so\n        every field is looked up tolerantly rather than assumed.\n        """\n        for n in names:\n            if n in df.columns:\n                return df[n]\n        return pd.Series([default] * len(df), index=df.index)\n\n    def _normalize_congress(self, df):\n        df = df.copy()\n        if "Range" in df.columns:\n            lows, highs = zip(*df["Range"].map(_parse_range)) if len(df) else ([], [])\n            amount_min, amount_max = list(lows), list(highs)\n        else:\n            amt = (self._col(df, "Amount", "Trade_Size_USD", "amount", default=0)\n                   .astype(str).str.replace(r"[$,]", "", regex=True))\n            amt = pd.to_numeric(amt, errors="coerce").fillna(0)\n            amount_min, amount_max = amt.tolist(), amt.tolist()\n        out = pd.DataFrame({\n            "ticker": self._col(df, "Ticker").astype(str).str.upper(),\n            "transaction_date": pd.to_datetime(self._col(df, "TransactionDate", "Traded"),\n                                               errors="coerce"),\n            "report_date": pd.to_datetime(self._col(df, "ReportDate", "Filed"),\n                                          errors="coerce"),\n            "representative": self._col(df, "Representative", "Name"),\n            "party": self._col(df, "Party"),\n            "chamber": self._col(df, "House", "Chamber"),\n            "transaction_type": self._col(df, "Transaction").astype(str).str.strip().str.title(),\n            "amount_min": amount_min,\n            "amount_max": amount_max,\n        })\n        return out\n\n    # ---- Insider transactions -------------------------------------------\n    def insider_trades(self):\n        raw = (mock_data.mock_insiders() if self.mock\n               else self._api.insiders())\n        return self._normalize_insiders(raw)\n\n    def _normalize_insiders(self, df):\n        df = df.copy()\n        code_map = {"P": "Purchase", "S": "Sale"}\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "transaction_date": pd.to_datetime(df["Date"]),\n            "insider_name": df["Name"],\n            "title": df["Title"],\n            "transaction_type": df["TransactionCode"].map(code_map).fillna("Other"),\n            "shares": df["Shares"].astype(int),\n            "price_per_share": df["PricePerShare"].astype(float),\n        })\n        out["value"] = (out["shares"] * out["price_per_share"]).round(2)\n        return out\n\n    # ---- Government contracts --------------------------------------------\n    def gov_contracts(self):\n        raw = (mock_data.mock_gov_contracts() if self.mock\n               else self._api.gov_contracts())\n        return self._normalize_gov(raw)\n\n    def _normalize_gov(self, df):\n        df = df.copy()\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "award_date": pd.to_datetime(df["Date"]),\n            "amount": df["Amount"].astype(float),\n            "agency": df["Agency"],\n        })\n        return out\n\n    # ---- Lobbying --------------------------------------------------------\n    def lobbying(self):\n        raw = (mock_data.mock_lobbying() if self.mock\n               else self._api.lobbying())\n        return self._normalize_lobbying(raw)\n\n    def _normalize_lobbying(self, df):\n        df = df.copy()\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "filing_date": pd.to_datetime(df["Date"]),\n            "amount": df["Amount"].astype(float),\n            "client": df["Client"],\n            "issue": df["Issue"],\n        })\n        return out\n')
open("src/quant_research/signals/congress.py", "w").write('"""Congressional cluster-buy signal.\n\nThe score rewards securities that several members of Congress are buying at once,\nwith each buy weighted by how informed it is likely to be. Features encode testable\nhypotheses; the backtest layer calibrates their weights.\n\nFeatures (each normalized across the cross-section, then weighted):\n  cluster              - distinct members buying the same name\n  size_vs_networth     - trade size relative to the member\'s net worth (conviction\n                         relative to means, controlling for the wealth confound)\n  committee_alignment  - buys by members whose committee oversees the ticker\'s sector\n  recency              - disclosures weighted toward the present\n  bipartisan           - a bonus when both parties are buying the same name\n\nMember net worth and committee membership come from the enrichment layer, which\nresolves a real implementation on live data and a synthetic one in mock mode.\n\nSignals are keyed on the DISCLOSURE (report) date, never the trade date, because a\ntrade is only actionable once it is public — which is what keeps the downstream\nbacktest free of look-ahead.\n"""\nfrom datetime import date\nimport numpy as np\nimport pandas as pd\n\nfrom .base import BaseSignal\nfrom ..enrichment.mock import MockEnrichment\nfrom ..enrichment.live import LiveEnrichment\n\nDEFAULT_WEIGHTS = {\n    "cluster": 0.30,\n    "size_vs_networth": 0.25,\n    "committee_alignment": 0.25,\n    "recency": 0.15,\n    "bipartisan": 0.05,\n}\n\n\nclass CongressSignal(BaseSignal):\n    name = "congress"\n    description = ("Clustered congressional purchases, weighted by conviction "\n                   "relative to net worth and by committee-sector alignment.")\n\n    def __init__(self, client, lookback_days=30, weights=None, enrichment=None):\n        self.client = client\n        self.lookback_days = lookback_days\n        self.weights = weights or dict(DEFAULT_WEIGHTS)\n        self.enrichment = enrichment or (\n            MockEnrichment() if getattr(client, "mock", False) else LiveEnrichment())\n\n    def compute(self, as_of=None, trades=None):\n        as_of = pd.Timestamp(as_of or date.today())\n        start = as_of - pd.Timedelta(days=self.lookback_days)\n\n        trades = self.client.congress_trades() if trades is None else trades\n        buys = trades[(trades.transaction_type == "Purchase")\n                      & (trades.report_date > start)\n                      & (trades.report_date <= as_of)].copy()\n        if buys.empty:\n            return pd.DataFrame(columns=["score"])\n\n        buys["mid_amount"] = (buys.amount_min + buys.amount_max) / 2.0\n\n        nw = buys.representative.map(self.enrichment.net_worth)\n        if nw.notna().any():\n            # conviction relative to means; unknown members fall back to the median\n            buys["conviction"] = buys.mid_amount / nw.fillna(nw.dropna().median())\n        else:\n            buys["conviction"] = buys.mid_amount      # no net worth available -> raw size\n\n        buys["days_ago"] = (as_of - buys.report_date).dt.days\n        buys["recency_w"] = np.select(\n            [buys.days_ago <= 7, buys.days_ago <= 14], [1.0, 0.6], default=0.3)\n        buys["aligned"] = [int(self.enrichment.is_aligned(r, t))\n                           for r, t in zip(buys.representative, buys.ticker)]\n\n        g = buys.groupby("ticker")\n        feat = pd.DataFrame({\n            "n_buys": g.size(),\n            "cluster": g.representative.nunique(),\n            "size_vs_networth": g.conviction.sum(),\n            "committee_alignment": g.aligned.sum(),\n            "recency": g.recency_w.sum(),\n            "bipartisan": g.party.apply(lambda p: int({"D", "R"}.issubset(set(p)))),\n        })\n\n        norm = {}\n        for f in ["cluster", "size_vs_networth", "committee_alignment", "recency"]:\n            col = feat[f].astype(float)\n            spread = col.max() - col.min()\n            norm[f] = (col - col.min()) / spread if spread > 0 else col * 0.0\n        norm["bipartisan"] = feat["bipartisan"].astype(float)\n        norm = pd.DataFrame(norm)\n\n        raw = sum(self.weights[f] * norm[f] for f in self.weights)\n        feat["score"] = self._scale_0_100(raw).round(1)\n        for f in norm.columns:\n            feat[f + "_n"] = norm[f].round(3)\n        return feat.sort_values("score", ascending=False)\n')
open("src/quant_research/enrichment/live.py", "w").write('"""Live enrichment from public sources.\n\n- Committee membership: the @unitedstates/congress-legislators project, fetched and\n  cached locally. Members are matched from a Quiver representative name to a bioguide\n  id, then to their current committee assignments.\n- Net worth: a curated reference table (reference_data/networth.csv), with a small\n  embedded fallback.\n- Ticker sector: a static map by default (no network), with an opt-in yfinance\n  fallback. The static path keeps the backtest fast and free of rate limits; the\n  fallback is available when fuller coverage is worth the API cost.\n\nUnmatched members and unmapped tickers degrade silently — the feature simply does\nnot fire, which never penalizes a name, it only declines to boost it.\n"""\nimport os\nimport re\nimport csv\nimport json\nimport unicodedata\n\nUS_BASE = "https://raw.githubusercontent.com/unitedstates/congress-legislators/main/"\nDEFAULT_CACHE = "data/reference"\nNETWORTH_CSV = "reference_data/networth.csv"\n\nSECTOR_RULES = [\n    ("armed services", {"Defense"}),\n    ("homeland security", {"Defense"}),\n    ("foreign affairs", {"Defense"}),\n    ("financial services", {"Financials"}),\n    ("banking", {"Financials"}),\n    ("ways and means", {"Financials"}),\n    ("energy", {"Energy", "Utilities"}),\n    ("natural resources", {"Energy"}),\n    ("commerce", {"Technology", "Consumer Discretionary"}),\n    ("science", {"Technology"}),\n    ("agriculture", {"Consumer Staples"}),\n    ("health", {"Health Care"}),\n    ("transportation", {"Industrials"}),\n    ("infrastructure", {"Industrials"}),\n]\n\n# static ticker -> sector, covering the large/mid caps Congress trades most. Sector\n# labels match SECTOR_RULES outputs. Extend freely; no network needed.\nTICKER_SECTOR = {\n    # Technology + communications (folded into Technology)\n    "AAPL": "Technology", "MSFT": "Technology", "NVDA": "Technology", "GOOGL": "Technology",\n    "GOOG": "Technology", "META": "Technology", "AVGO": "Technology", "ORCL": "Technology",\n    "CRM": "Technology", "ADBE": "Technology", "CSCO": "Technology", "INTC": "Technology",\n    "AMD": "Technology", "QCOM": "Technology", "TXN": "Technology", "IBM": "Technology",\n    "NOW": "Technology", "INTU": "Technology", "AMAT": "Technology", "MU": "Technology",\n    "PANW": "Technology", "PLTR": "Technology", "NFLX": "Technology", "TMUS": "Technology",\n    "CMCSA": "Technology", "T": "Technology", "VZ": "Technology",\n    # Defense / aerospace\n    "LMT": "Defense", "RTX": "Defense", "NOC": "Defense", "GD": "Defense", "LHX": "Defense",\n    "AXON": "Defense", "BA": "Defense", "HII": "Defense",\n    # Financials\n    "JPM": "Financials", "BAC": "Financials", "WFC": "Financials", "C": "Financials",\n    "GS": "Financials", "MS": "Financials", "BLK": "Financials", "SCHW": "Financials",\n    "AXP": "Financials", "V": "Financials", "MA": "Financials", "COF": "Financials",\n    "USB": "Financials", "PNC": "Financials", "BX": "Financials", "SPGI": "Financials",\n    # Health Care\n    "UNH": "Health Care", "JNJ": "Health Care", "LLY": "Health Care", "PFE": "Health Care",\n    "ABBV": "Health Care", "MRK": "Health Care", "TMO": "Health Care", "ABT": "Health Care",\n    "DHR": "Health Care", "BMY": "Health Care", "AMGN": "Health Care", "GILD": "Health Care",\n    "CVS": "Health Care", "MDT": "Health Care", "ISRG": "Health Care", "VRTX": "Health Care",\n    # Energy\n    "XOM": "Energy", "CVX": "Energy", "COP": "Energy", "SLB": "Energy", "EOG": "Energy",\n    "OXY": "Energy", "PSX": "Energy", "MPC": "Energy", "VLO": "Energy", "KMI": "Energy",\n    # Utilities\n    "NEE": "Utilities", "DUK": "Utilities", "SO": "Utilities", "D": "Utilities",\n    "AEP": "Utilities", "EXC": "Utilities", "SRE": "Utilities",\n    # Consumer Staples\n    "WMT": "Consumer Staples", "COST": "Consumer Staples", "PG": "Consumer Staples",\n    "KO": "Consumer Staples", "PEP": "Consumer Staples", "MO": "Consumer Staples",\n    "PM": "Consumer Staples", "CL": "Consumer Staples", "MDLZ": "Consumer Staples",\n    "TGT": "Consumer Staples", "CELH": "Consumer Staples",\n    # Consumer Discretionary\n    "AMZN": "Consumer Discretionary", "TSLA": "Consumer Discretionary", "HD": "Consumer Discretionary",\n    "MCD": "Consumer Discretionary", "SBUX": "Consumer Discretionary", "NKE": "Consumer Discretionary",\n    "DIS": "Consumer Discretionary", "BKNG": "Consumer Discretionary", "LOW": "Consumer Discretionary",\n    "TJX": "Consumer Discretionary", "F": "Consumer Discretionary", "GM": "Consumer Discretionary",\n    "MELI": "Consumer Discretionary", "CAVA": "Consumer Discretionary",\n    # Industrials\n    "GE": "Industrials", "CAT": "Industrials", "HON": "Industrials", "DE": "Industrials",\n    "UNP": "Industrials", "UPS": "Industrials", "MMM": "Industrials", "EMR": "Industrials",\n    "ETN": "Industrials", "ITW": "Industrials", "CSX": "Industrials", "NSC": "Industrials",\n}\n\n_YF_SECTOR = {\n    "Technology": "Technology", "Industrials": "Industrials",\n    "Financial Services": "Financials", "Healthcare": "Health Care",\n    "Energy": "Energy", "Utilities": "Utilities",\n    "Consumer Defensive": "Consumer Staples", "Consumer Cyclical": "Consumer Discretionary",\n    "Communication Services": "Technology",\n}\n\n_NETWORTH_FALLBACK = {\n    "P000197": 120_000_000, "T000278": 6_000_000, "K000389": 30_000_000,\n    "M001157": 100_000_000, "C001120": 3_000_000,\n}\n\n_SUFFIX = {"jr", "sr", "ii", "iii", "iv", "v"}\n\n\ndef _norm(s):\n    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()\n    return re.sub(r"\\s+", " ", re.sub(r"[^a-z ]", " ", s.lower())).strip()\n\n\ndef _firstlast(s):\n    toks = [t for t in _norm(s).split() if t not in _SUFFIX]\n    return f"{toks[0]} {toks[-1]}" if len(toks) >= 2 else _norm(s)\n\n\ndef _committee_sectors(committee_name):\n    out, low = set(), committee_name.lower()\n    for kw, secs in SECTOR_RULES:\n        if kw in low:\n            out |= secs\n    return out\n\n\nclass LiveEnrichment:\n    def __init__(self, cache_dir=DEFAULT_CACHE, networth_csv=NETWORTH_CSV,\n                 use_yf_sectors=False):\n        self.cache_dir = cache_dir\n        self.networth_csv = networth_csv\n        self.use_yf_sectors = use_yf_sectors    # opt-in network sector lookups\n        self._name_to_bio = {}\n        self._bio_committees = {}\n        self._networth = {}\n        self._sector_cache = {}\n        self._load()\n\n    def _fetch_yaml(self, filename):\n        import requests\n        import yaml\n        os.makedirs(self.cache_dir, exist_ok=True)\n        cache = os.path.join(self.cache_dir, filename.replace(".yaml", ".json"))\n        if os.path.exists(cache):\n            with open(cache) as f:\n                return json.load(f)\n        data = yaml.safe_load(requests.get(US_BASE + filename, timeout=120).text)\n        with open(cache, "w") as f:\n            json.dump(data, f)\n        return data\n\n    def _load(self):\n        legislators = self._fetch_yaml("legislators-current.yaml")\n        committees = self._fetch_yaml("committees-current.yaml")\n        membership = self._fetch_yaml("committee-membership-current.yaml")\n        for p in legislators:\n            n, bio = p["name"], p["id"]["bioguide"]\n            variants = {f"{n.get(\'first\',\'\')} {n.get(\'last\',\'\')}", n.get("official_full", "")}\n            if n.get("nickname"):\n                variants.add(f"{n[\'nickname\']} {n.get(\'last\',\'\')}")\n            for v in variants:\n                if v.strip():\n                    self._name_to_bio[_norm(v)] = bio\n                    self._name_to_bio[_firstlast(v)] = bio\n        comm_name = {c["thomas_id"]: c["name"] for c in committees}\n        for cid, members in membership.items():\n            if len(cid) == 4:\n                name = comm_name.get(cid, cid)\n                for m in members:\n                    self._bio_committees.setdefault(m["bioguide"], set()).add(name)\n        self._networth = self._load_networth()\n\n    def _load_networth(self):\n        if os.path.exists(self.networth_csv):\n            out = {}\n            with open(self.networth_csv) as f:\n                for row in csv.DictReader(f):\n                    bio = row.get("bioguide", "").strip()\n                    val = row.get("net_worth", "").strip()\n                    if bio and val:\n                        out[bio] = float(val)\n            if out:\n                return out\n        return dict(_NETWORTH_FALLBACK)\n\n    def resolve(self, representative):\n        return (self._name_to_bio.get(_norm(representative))\n                or self._name_to_bio.get(_firstlast(representative)))\n\n    def net_worth(self, representative):\n        return self._networth.get(self.resolve(representative))\n\n    def committee_sectors(self, representative):\n        comms = self._bio_committees.get(self.resolve(representative), set())\n        out = set()\n        for c in comms:\n            out |= _committee_sectors(c)\n        return out\n\n    def sector_of(self, ticker):\n        if ticker in TICKER_SECTOR:\n            return TICKER_SECTOR[ticker]\n        if not self.use_yf_sectors:\n            return None\n        if ticker in self._sector_cache:\n            return self._sector_cache[ticker]\n        sector = None\n        try:\n            import yfinance as yf\n            info = yf.Ticker(ticker.replace(".", "-")).info\n            sector = _YF_SECTOR.get(info.get("sector"))\n        except Exception:\n            sector = None\n        self._sector_cache[ticker] = sector\n        return sector\n\n    def prime_sectors(self, tickers):\n        for t in tickers:\n            self.sector_of(t)\n        return sum(1 for t in tickers if self.sector_of(t) is not None)\n\n    def is_aligned(self, representative, ticker):\n        sec = self.sector_of(ticker)\n        return sec is not None and sec in self.committee_sectors(representative)\n')
print("updated ingestion, signal, and enrichment layers")

updated ingestion, signal, and enrichment layers


In [4]:
open("src/quant_research/backtest/__init__.py", "w").write('"""Event-study backtesting and risk metrics."""\n')
open("src/quant_research/backtest/prices.py", "w").write('"""Price data and forward returns.\n\nThe price layer is kept separate from the backtest engine: it fetches adjusted\nclose prices (via yfinance) into a wide DataFrame, and the engine consumes that\nDataFrame. This separation lets the engine\'s logic be tested with synthetic prices\nthat have a known relationship to the signal.\n"""\nimport pandas as pd\n\n\ndef price_history(tickers, start, end):\n    """Wide DataFrame of adjusted close: index = trading day, columns = tickers.\n\n    Share-class tickers are translated from the dotted convention (BRK.B) to the\n    dashed one yfinance expects (BRK-B), then renamed back so columns align with the\n    signal\'s ticker index. Fully-empty columns (delisted or unfetchable names) are\n    dropped, since they carry no events.\n    """\n    import yfinance as yf\n    tickers = list(dict.fromkeys(tickers))            # de-dup, preserve order\n    yf_map = {t: t.replace(".", "-") for t in tickers}\n    data = yf.download(list(yf_map.values()), start=start, end=end,\n                       auto_adjust=True, progress=False)\n    if isinstance(data.columns, pd.MultiIndex):\n        close = data["Close"].rename(columns={v: k for k, v in yf_map.items()})\n    else:\n        close = data[["Close"]].rename(columns={"Close": tickers[0]})\n    return close.dropna(how="all").dropna(axis=1, how="all")\n\n\ndef forward_returns(prices, as_of, horizon):\n    """Return over `horizon` trading days after `as_of`, per ticker.\n\n    Entry is the first trading day on or after the signal date; exit is `horizon`\n    trading days later. Names without enough future data return NaN and are dropped\n    downstream, which avoids any look-ahead at the end of the sample.\n    """\n    idx = prices.index\n    pos = idx.searchsorted(pd.Timestamp(as_of))\n    if pos >= len(idx) or pos + horizon >= len(idx):\n        return pd.Series(dtype=float, index=prices.columns)\n    entry = prices.iloc[pos]\n    exit_ = prices.iloc[pos + horizon]\n    return (exit_ / entry) - 1.0\n')
open("src/quant_research/backtest/engine.py", "w").write('"""Event-study backtester for a cross-sectional signal.\n\nOn each rebalance date the engine scores the universe, sorts names into buckets by\nscore, and measures their forward return over a fixed horizon. Aggregated across\ndates, this answers the core question: do higher-scoring names earn higher forward\nreturns? It reports three complementary views:\n\n  - bucket means: average forward return by score bucket (monotonic is the goal)\n  - information coefficient (IC): rank correlation of score with forward return,\n    averaged across dates, with the share of dates where it is positive\n  - a long-top-bucket equity curve with standard risk metrics\n\nThe engine takes a `score_fn(date) -> Series[ticker -> score]`, so it is agnostic to\nwhich signal it is testing.\n"""\nimport numpy as np\nimport pandas as pd\n\nfrom .prices import forward_returns\n\n\ndef _spearman(a, b):\n    if a.nunique() < 2 or b.nunique() < 2:\n        return np.nan\n    return a.rank().corr(b.rank())\n\n\ndef event_study(score_fn, prices, rebalance_dates, horizon=21, n_buckets=5):\n    bucket_rows, ic_rows, ls_rows = [], [], []\n    for d in rebalance_dates:\n        scores = score_fn(d)\n        if scores is None or len(scores) < n_buckets:\n            continue\n        fwd = forward_returns(prices, d, horizon)\n        df = pd.concat([scores.rename("score"), fwd.rename("fwd")], axis=1).dropna()\n        if len(df) < n_buckets:\n            continue\n        df["bucket"] = pd.qcut(df["score"].rank(method="first"), n_buckets, labels=False)\n        means = df.groupby("bucket")["fwd"].mean()\n        for b, m in means.items():\n            bucket_rows.append({"date": d, "bucket": int(b), "fwd": m})\n        ls_rows.append({"date": d, "long_short": means.get(n_buckets - 1) - means.get(0)})\n        ic_rows.append({"date": d, "ic": _spearman(df["score"], df["fwd"])})\n\n    buckets = pd.DataFrame(bucket_rows)\n    ls = pd.DataFrame(ls_rows)\n    ic = pd.DataFrame(ic_rows).dropna()\n\n    summary = {\n        "n_periods": len(ls),\n        "mean_ic": ic["ic"].mean() if len(ic) else np.nan,\n        "ic_positive_share": (ic["ic"] > 0).mean() if len(ic) else np.nan,\n        "mean_long_short": ls["long_short"].mean() if len(ls) else np.nan,\n        "ls_positive_share": (ls["long_short"] > 0).mean() if len(ls) else np.nan,\n    }\n    bucket_means = (buckets.groupby("bucket")["fwd"].mean()\n                    if len(buckets) else pd.Series(dtype=float))\n    return {"summary": summary, "bucket_means": bucket_means,\n            "long_short": ls, "ic": ic}\n\n\ndef feature_buckets(score_fn_full, prices, rebalance_dates, feature, horizon=21):\n    """Forward return split by whether a 0/1 feature is present (e.g. committee_alignment>0)."""\n    on, off = [], []\n    for d in rebalance_dates:\n        panel = score_fn_full(d)              # full feature DataFrame indexed by ticker\n        if panel is None or panel.empty:\n            continue\n        fwd = forward_returns(prices, d, horizon)\n        df = panel.join(fwd.rename("fwd")).dropna(subset=["fwd"])\n        if df.empty:\n            continue\n        flag = df[feature] > 0\n        on.extend(df.loc[flag, "fwd"].tolist())\n        off.extend(df.loc[~flag, "fwd"].tolist())\n    return {"with_feature_mean": np.mean(on) if on else np.nan, "with_n": len(on),\n            "without_feature_mean": np.mean(off) if off else np.nan, "without_n": len(off)}\n\n\ndef long_top_curve(score_fn, prices, rebalance_dates, horizon=21, top_frac=0.2):\n    """Equity curve of equal-weighting the top score fraction on a non-overlapping\n    schedule. Entries are kept at least `horizon` trading days apart, measured against\n    the price index, so the cadence is correct whatever the rebalance-date frequency."""\n    idx = prices.index\n    rets, last_pos = [], -horizon\n    for d in rebalance_dates:\n        pos = idx.searchsorted(pd.Timestamp(d))\n        if pos - last_pos < horizon:          # keep holding periods from overlapping\n            continue\n        scores = score_fn(d)\n        if scores is None or len(scores) == 0:\n            continue\n        fwd = forward_returns(prices, d, horizon)\n        df = pd.concat([scores.rename("score"), fwd.rename("fwd")], axis=1).dropna()\n        if df.empty:\n            continue\n        k = max(1, int(round(len(df) * top_frac)))\n        rets.append(df.nlargest(k, "score")["fwd"].mean())\n        last_pos = pos\n    rets = pd.Series(rets)\n    equity = (1 + rets).cumprod()\n    return equity, rets\n\n\ndef long_short_curve(score_fn, prices, rebalance_dates, horizon=21, top_frac=0.2):\n    """Market-neutral equity curve: long the top score fraction, short the bottom,\n    on a non-overlapping schedule. Isolates the signal from market beta."""\n    idx = prices.index\n    rets, last_pos = [], -horizon\n    for d in rebalance_dates:\n        pos = idx.searchsorted(pd.Timestamp(d))\n        if pos - last_pos < horizon:\n            continue\n        scores = score_fn(d)\n        if scores is None or len(scores) == 0:\n            continue\n        df = pd.concat([scores.rename("score"),\n                        forward_returns(prices, d, horizon).rename("fwd")], axis=1).dropna()\n        if len(df) < 2:\n            continue\n        k = max(1, int(round(len(df) * top_frac)))\n        rets.append(df.nlargest(k, "score")["fwd"].mean() - df.nsmallest(k, "score")["fwd"].mean())\n        last_pos = pos\n    rets = pd.Series(rets)\n    return (1 + rets).cumprod(), rets\n\n\ndef benchmark_curve(prices, rebalance_dates, horizon=21):\n    """Equal-weight all available names each non-overlapping period: a universe\n    buy-and-hold proxy for separating signal alpha from market beta."""\n    idx = prices.index\n    rets, last_pos = [], -horizon\n    for d in rebalance_dates:\n        pos = idx.searchsorted(pd.Timestamp(d))\n        if pos - last_pos < horizon:\n            continue\n        fwd = forward_returns(prices, d, horizon).dropna()\n        if fwd.empty:\n            continue\n        rets.append(fwd.mean())\n        last_pos = pos\n    rets = pd.Series(rets)\n    return (1 + rets).cumprod(), rets\n\n\ndef metrics(rets, horizon=21):\n    """Annualized risk/return metrics from a series of per-period returns."""\n    if len(rets) == 0:\n        return {"cagr": float("nan"), "sharpe": float("nan"), "sortino": float("nan"),\n                "max_drawdown": float("nan"), "hit_rate": float("nan"), "n_trades": 0}\n    per_year = 252 / horizon\n    growth = (1 + rets).prod()\n    cagr = growth ** (per_year / len(rets)) - 1\n    vol = rets.std(ddof=1)\n    sharpe = (rets.mean() / vol * np.sqrt(per_year)) if vol > 0 else np.nan\n    downside = rets[rets < 0].std(ddof=1)\n    sortino = (rets.mean() / downside * np.sqrt(per_year)) if downside and downside > 0 else np.nan\n    equity = (1 + rets).cumprod()\n    max_dd = (equity / equity.cummax() - 1).min()\n    return {"cagr": cagr, "sharpe": sharpe, "sortino": sortino,\n            "max_drawdown": max_dd, "hit_rate": (rets > 0).mean(), "n_trades": len(rets)}\n')
open("tests/test_backtest.py", "w").write('"""The engine must detect edge when present and report null on random scores."""\nimport numpy as np\nimport pandas as pd\nfrom quant_research.backtest.engine import event_study\n\n\ndef _prices(seed=0, n_tickers=40, n_days=400):\n    rng = np.random.default_rng(seed)\n    dates = pd.bdate_range("2021-01-01", periods=n_days)\n    steps = rng.normal(0, 0.01, size=(n_days, n_tickers))\n    cols = [f"T{i:02d}" for i in range(n_tickers)]\n    return pd.DataFrame(100 * np.exp(np.cumsum(steps, 0)), index=dates, columns=cols), rng\n\n\ndef test_engine_detects_edge():\n    prices, rng = _prices()\n    rebal = pd.bdate_range("2021-02-01", "2022-06-01", freq="W-FRI")\n    H = 21\n\n    def score_edge(d):\n        pos = prices.index.searchsorted(pd.Timestamp(d))\n        if pos + H >= len(prices.index):\n            return None\n        fwd = prices.iloc[pos + H] / prices.iloc[pos] - 1\n        return pd.Series(fwd + rng.normal(0, 0.02, len(fwd)), index=prices.columns)\n\n    s = event_study(score_edge, prices, rebal, horizon=H)["summary"]\n    assert s["mean_ic"] > 0.3\n    assert s["mean_long_short"] > 0\n\n\ndef test_engine_reports_null_on_noise():\n    prices, rng = _prices(seed=1)\n    rebal = pd.bdate_range("2021-02-01", "2022-06-01", freq="W-FRI")\n    s = event_study(lambda d: pd.Series(rng.normal(size=prices.shape[1]),\n                                        index=prices.columns),\n                    prices, rebal, horizon=21)["summary"]\n    assert abs(s["mean_ic"]) < 0.15\n')
print("wrote backtest layer and tests")

wrote backtest layer and tests


In [5]:
!pip install -q -e .

import os, sys, importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import quant_research; importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.1.0


## Step 1 — Validate the engine before trusting it

Before backtesting the real signal, we confirm the engine behaves on data where the
answer is known: random-walk prices with one score that genuinely predicts the
next-horizon return and one that is pure noise. A trustworthy engine reports strong
edge for the first and nothing for the second.


In [6]:
import numpy as np, pandas as pd
from quant_research.backtest.engine import event_study

rng = np.random.default_rng(0)
tk = [f"T{i:02d}" for i in range(40)]
dates = pd.bdate_range("2021-01-01", periods=600)
synth = pd.DataFrame(100*np.exp(np.cumsum(rng.normal(0,0.01,(len(dates),len(tk))),0)),
                     index=dates, columns=tk)
rebal = pd.bdate_range("2021-02-01", "2022-12-01", freq="W-FRI"); H = 21

def score_edge(d):
    pos = synth.index.searchsorted(pd.Timestamp(d))
    if pos+H >= len(synth.index): return None
    fwd = synth.iloc[pos+H]/synth.iloc[pos]-1
    return pd.Series(fwd + rng.normal(0,0.02,len(fwd)), index=synth.columns)

for label, fn in [("edge present", score_edge),
                  ("random noise", lambda d: pd.Series(rng.normal(size=len(tk)), index=tk))]:
    r = event_study(fn, synth, rebal, horizon=H)["summary"]
    print(f"{label:14s} | mean IC {r['mean_ic']:+.3f} | long-short {r['mean_long_short']:+.4f} "
          f"| LS positive {r['ls_positive_share']:.0%}")

edge present   | mean IC +0.896 | long-short +0.1109 | LS positive 100%
random noise   | mean IC +0.001 | long-short +0.0014 | LS positive 54%


A high IC and a consistently positive spread for the real signal, and effectively
zero for noise. That null result is the important one: it shows the engine is not
quietly using future information.


## Step 2 — Backtest the congress signal

We pull the trade history once, fetch prices for the names that appear, and score the
universe on weekly rebalance dates. `START_DATE` narrows the window to recent, denser
years; widen it to study the full history.

A note on the mock run: the synthetic trades carry a built-in tilt toward a few
tickers, so any apparent edge there mostly reflects how those names happened to move.
The genuine measurement comes from the live run.


In [7]:
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.congress import CongressSignal
from quant_research.enrichment.live import LiveEnrichment
from quant_research.backtest import prices as P, engine as E

START_DATE = "2022-01-01"     # widen to study more history
UNIVERSE_CAP = 120            # most-traded names, capped for a tractable price pull
HORIZON = 21                  # ~1 trading month forward

if USE_LIVE:
    client = QuiverClient(token=QUIVER_TOKEN)
    trades = client.congress_trades(historical=True)
else:
    client = QuiverClient(mock=True, mock_history_days=540)
    trades = client.congress_trades()

enrichment = LiveEnrichment()    # static sectors, no per-ticker network calls
signal = CongressSignal(client, lookback_days=30, enrichment=enrichment)

trades = trades[trades.report_date >= pd.Timestamp(START_DATE)]
buys = trades[trades.transaction_type == "Purchase"]
counts = buys.ticker.value_counts()
universe = counts[counts >= 3].index.tolist()[:UNIVERSE_CAP]

start = max(buys.report_date.min().date(), pd.Timestamp(START_DATE).date())
end = buys.report_date.max().date()
print(f"{len(trades)} trades | {len(universe)} tickers | {start} to {end}")

price_panel = P.price_history(universe, start, end)
if price_panel.empty or price_panel.shape[1] == 0:
    print("\n[!] price panel is empty — yfinance may be rate-limiting.")
    print("    Wait a few minutes (or restart the runtime), then re-run this cell.")
else:
    print("price panel:", price_panel.shape)

https://api.quiverquant.com/beta/bulk/congresstrading
45547 trades | 120 tickers | 2022-01-01 to 2026-06-16
price panel: (1116, 118)


In [8]:
rebalance_dates = pd.bdate_range(start, end, freq="W-FRI")
score_fn = lambda d: signal.compute(as_of=d, trades=trades)["score"]

result = E.event_study(score_fn, price_panel, rebalance_dates, horizon=HORIZON, n_buckets=5)
s = result["summary"]
if s["n_periods"] == 0:
    print("[!] no periods had enough data — confirm the price panel populated above.")
else:
    print(f"periods tested: {s['n_periods']}")
    print(f"mean IC: {s['mean_ic']:+.3f}  (positive {s['ic_positive_share']:.0%} of periods)")
    print(f"mean long-short ({HORIZON}d): {s['mean_long_short']:+.3%}  "
          f"(positive {s['ls_positive_share']:.0%} of periods)")

periods tested: 227
mean IC: +0.017  (positive 53% of periods)
mean long-short (21d): +0.727%  (positive 58% of periods)


### Forward return by score bucket

If the score carries information, the bars rise from the lowest bucket to the
highest. A flat or jagged profile says the score is not separating winners from
losers over this window.


In [9]:
import plotly.express as px

bm = result["bucket_means"]
if bm.empty:
    print("no bucket data to plot")
else:
    df = bm.reset_index(); df.columns = ["bucket", "mean_fwd_return"]
    df["bucket"] = df["bucket"].map({0:"1 (lowest)",1:"2",2:"3",3:"4",4:"5 (highest)"})
    fig = px.bar(df, x="bucket", y="mean_fwd_return",
                 title=f"Mean {HORIZON}-day forward return by congress-score bucket",
                 labels={"mean_fwd_return": "mean forward return"})
    fig.update_traces(marker_color="#2563eb")
    fig.update_layout(height=380, yaxis_tickformat=".1%")
    fig.show()

### Strategy view — separating alpha from beta

Three curves on a non-overlapping monthly schedule. **Long-only top bucket** holds the
top-scoring names; its returns mix the signal with full market exposure. **Universe**
equal-weights every name, a buy-and-hold proxy for that market exposure. **Long-short**
holds the top bucket and shorts the bottom, which cancels most of the market and leaves
the signal's own contribution. Comparing the three shows how much of the long-only
result is the score versus the market it rode.


In [10]:
import plotly.graph_objects as go

eq_top, rets_top = E.long_top_curve(score_fn, price_panel, rebalance_dates, horizon=HORIZON, top_frac=0.2)
eq_ls,  rets_ls  = E.long_short_curve(score_fn, price_panel, rebalance_dates, horizon=HORIZON, top_frac=0.2)
eq_bm,  _        = E.benchmark_curve(price_panel, rebalance_dates, horizon=HORIZON)

m_top, m_ls = E.metrics(rets_top, HORIZON), E.metrics(rets_ls, HORIZON)
if m_top["n_trades"] == 0:
    print("[!] no completed trades — the backtest produced no events (see the price-panel note above).")
else:
    print(f"long-only top : CAGR {m_top['cagr']:+.1%} | Sharpe {m_top['sharpe']:.2f} "
          f"| maxDD {m_top['max_drawdown']:.1%} | hit {m_top['hit_rate']:.0%} | n {m_top['n_trades']}")
    print(f"long-short    : CAGR {m_ls['cagr']:+.1%} | Sharpe {m_ls['sharpe']:.2f} "
          f"| maxDD {m_ls['max_drawdown']:.1%} | hit {m_ls['hit_rate']:.0%} | n {m_ls['n_trades']}")
    fig = go.Figure()
    fig.add_scatter(y=eq_top.values, name="Top bucket (long-only)", line=dict(color="#16a34a"))
    fig.add_scatter(y=eq_bm.values,  name="Universe (equal-weight)", line=dict(color="#9ca3af", dash="dash"))
    fig.add_scatter(y=eq_ls.values,  name="Long-short (market-neutral)", line=dict(color="#2563eb"))
    fig.update_layout(title="Equity curves (growth of 1)", height=400,
                      xaxis_title="period", yaxis_title="equity")
    fig.show()

long-only top : CAGR +9.5% | Sharpe 0.47 | maxDD -30.7% | hit 57% | n 46
long-short    : CAGR +12.9% | Sharpe 0.79 | maxDD -13.1% | hit 61% | n 46


### Feature attribution — does committee alignment pay?

This is how a feature earns its weight. We split every scored name by whether its
buying was committee-aligned and compare forward returns. A positive gap argues for
weight; a gap near zero argues for trimming it.


In [11]:
panel_fn = lambda d: signal.compute(as_of=d, trades=trades)
attr = E.feature_buckets(panel_fn, price_panel, rebalance_dates,
                         feature="committee_alignment", horizon=HORIZON)
if attr["with_n"] == 0 and attr["without_n"] == 0:
    print("no events available for feature attribution")
else:
    print(f"committee-aligned buys : mean {HORIZON}d return {attr['with_feature_mean']:+.3%}  "
          f"(n={attr['with_n']})")
    print(f"non-aligned buys       : mean {HORIZON}d return {attr['without_feature_mean']:+.3%}  "
          f"(n={attr['without_n']})")

committee-aligned buys : mean 21d return +2.176%  (n=1020)
non-aligned buys       : mean 21d return +1.228%  (n=14662)


## Reading the results honestly

- **The IC and the long-short consistency matter more than any single number.** A
  modest IC (say $0.03$–$0.06$) that is positive in a clear majority of periods is a
  real, tradeable signal. A large figure from a handful of periods is usually noise.
- **Read the long-short, not the long-only, for the signal's own edge.** The long-only
  top bucket carries full market exposure, so its Sharpe and drawdown mostly describe
  the market it rode. The long-short curve cancels that beta and shows what the score
  itself contributed.
- **Costs and regime still apply.** The curves exclude transaction costs and assume
  clean fills, and an edge over this window may not persist; the honest output is a
  confidence range with caveats.
- **Coverage shapes the committee feature.** With static sectors, alignment fires for
  the mapped large caps; enabling the yfinance fallback (`LiveEnrichment(use_yf_sectors=True)`)
  broadens it at the cost of API calls.

This backtest is also what turns the feature weights from priors into evidence: the
bucket profile and feature attribution show which features to up-weight and which to trim.


## Commit

We commit the backtest layer and its tests.


In [12]:
!git config --global user.email "aballa1234@gmail.com"
!git config --global user.name "Alex Balla"

!git add -A
!git commit -m "Add event-study backtest engine, price layer, and tests"
!git push

[main 061e046] Add event-study backtest engine, price layer, and tests
 8 files changed, 358 insertions(+), 52 deletions(-)
 create mode 100644 src/quant_research/backtest/engine.py
 create mode 100644 src/quant_research/backtest/prices.py
 create mode 100644 tests/test_backtest.py
Enumerating objects: 30, done.
Counting objects: 100% (30/30), done.
Delta compression using up to 2 threads
Compressing objects: 100% (15/15), done.
Writing objects: 100% (17/17), 7.10 KiB | 1.77 MiB/s, done.
Total 17 (delta 6), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (6/6), completed with 6 local objects.
To https://github.com/balla-a12/quant-equity-research.git
   5846ebe..061e046  main -> main


## Recap and next

The project now measures, not just ranks. The event-study engine reports whether the
congress score precedes returns, with an information coefficient, a bucket profile, a
strategy equity curve, and feature attribution that calibrates the weights — validated
against synthetic data where the answer is known.

Next, the remaining signals (government contracts, lobbying, off-exchange) join on the
same `BaseSignal` contract, the composite blends them, and the play classifier maps a
composite score and volatility conditions to a trade structure. The dashboard in
notebook `03` surfaces today's candidates with the historical edge behind each.
